# Repeated beta1 ROC simulation with a lambda1 path

This notebook holds $\lambda_2$ fixed and varies $\lambda_1$. For each signal level and each lambda1 value, it averages the FPR and TPR coordinates of the converged replications, then connects those mean points in lambda1 order. No artificial endpoints are added.

`TARGET_SIGNAL_FRACTIONS` specifies $1-\operatorname{Var}(epsilon)/\operatorname{Var}(y)$. The notebook converts these targets to the simulator's required $\mathrm{SNR}=\operatorname{sd}(signal)/\operatorname{sd}(epsilon)$. Lambda2 is fixed while lambda1 varies; beta1 is the primary ROC target. True beta values are assigned by the standard Schaefer 2018 200-parcel, 7-network order; your connectivity arrays must use that same order.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

MODEL_DIR = Path.cwd() / 'model'
if not (MODEL_DIR / 'roc_experiment.py').exists():
    MODEL_DIR = Path.cwd()
sys.path.insert(0, str(MODEL_DIR))

from config import PARAMS
from atlas_utils import betas_from_schaefer_parcel_selections
from roc_experiment import (
    RESULTS_DIR, run_beta1_roc_experiment,
    plot_beta1_centroid_path,
    fit_beta2_roc_detail, plot_beta2_roc_detail,
)


In [ ]:
# Schaefer 2018, 200 parcels, 7 networks. Edit these sparse selections.
# roi_indices are zero-based and must belong to the stated network.
# Valid names: Vis, SomMot, DorsAttn, SalVentAttn, Limbic, Cont, Default.
PARCEL_SELECTIONS = [
    {'network': 'SomMot', 'roi_indices': [19, 20, 21, 22, 23], 'beta1': -8},
    {'network': 'SomMot', 'roi_indices': [24, 25, 26, 27, 28], 'beta2': -6},
    {'network': 'Cont', 'roi_indices': [60, 61, 62, 63, 64], 'beta1': -4},
    {'network': 'Cont', 'roi_indices': [68, 69, 70, 71, 72], 'beta2': 6},
    {'network': 'Default', 'roi_indices': [79, 80, 81, 82, 83], 'beta1': 6},
    {'network': 'Default', 'roi_indices': [95, 96, 97, 98, 99], 'beta2': 10},
]

# The plotted beta1 path connects the mean point for each lambda1.
FIXED_LAMBDA2 = 100
LAMBDA1_VALUES = [100, 150, 200]
# Target 1 - Var(epsilon) / Var(y), not the simulator's direct SNR.
TARGET_SIGNAL_FRACTIONS = [0.6, 0.7, 0.8]
SNR_VALUES = [np.sqrt(value / (1 - value)) for value in TARGET_SIGNAL_FRACTIONS]
N_REPLICATES = 10
BASE_SEED = 2026

# These overrides affect this ROC experiment only.
MAX_ITER = 6000
N_JOBS = 3

print(f'beta1 ROC: lambda2={FIXED_LAMBDA2}; lambda1={LAMBDA1_VALUES}; '
      f'target 1 - Var(epsilon)/Var(y)={TARGET_SIGNAL_FRACTIONS}; '
      f'simulator SNR={[round(value, 4) for value in SNR_VALUES]}; '
      f'replications={N_REPLICATES}')


In [ ]:
beta1_true, beta2_true, assignment_table = betas_from_schaefer_parcel_selections(
    PARCEL_SELECTIONS, p=PARAMS['P']
)
display(assignment_table)

run_dir, roc_points, beta1_centers = run_beta1_roc_experiment(
    name='beta1_roc_lambda2_100',
    data_dir=PARAMS['DATA_DIR'],
    p=PARAMS['P'],
    n=PARAMS['N_SUBJECTS'],
    beta1_true=beta1_true,
    beta2_true=beta2_true,
    snr_values=SNR_VALUES,
    lambda2=FIXED_LAMBDA2,
    lambda1_values=LAMBDA1_VALUES,
    n_replicates=N_REPLICATES,
    seed=BASE_SEED,
    max_iter=MAX_ITER,
    step_size=PARAMS['STEP_SIZE'],
    tol=PARAMS['TOL'],
    num_candidates=PARAMS['NUM_CANDIDATES'],
    initialization=PARAMS['INITIALIZATION'],
    n_jobs=N_JOBS,
    results_dir=RESULTS_DIR,
)
print(f'Saved ROC experiment to {run_dir}')
display(roc_points.groupby('snr').agg(
    fits=('converged', 'size'), converged=('converged', 'sum'),
    mean_stationarity=('stationarity', 'mean'),
))


In [ ]:
target_by_snr = dict(zip(SNR_VALUES, TARGET_SIGNAL_FRACTIONS))
beta1_centers['target_signal_fraction'] = beta1_centers['snr'].map(target_by_snr)
display(beta1_centers)  # n_replicates shows how many converged fits formed each center.
fig, ax = plot_beta1_centroid_path(
    beta1_centers, lambda1_values=LAMBDA1_VALUES,
    save_path=run_dir / 'figures' / 'beta1_centroid_path.png',
    label_column='target_signal_fraction',
    label_prefix='1 - Var(epsilon)/Var(y)',
)
plt.show()


## Inspect one ROC-path point

Choose one target signal fraction, replication, and lambda1 value. This refits that one model using the same random seed as the ROC experiment and plots its true and estimated beta vectors.

In [ ]:
DETAIL_TARGET_SIGNAL_FRACTION = 0.7
DETAIL_REPLICATE = 0  # Zero-based: 0 is the first replication.
DETAIL_LAMBDA1 = 100

if DETAIL_TARGET_SIGNAL_FRACTION not in TARGET_SIGNAL_FRACTIONS:
    raise ValueError('Choose a value from TARGET_SIGNAL_FRACTIONS.')
if DETAIL_LAMBDA1 not in LAMBDA1_VALUES:
    raise ValueError('Choose a value from LAMBDA1_VALUES.')
snr_index = TARGET_SIGNAL_FRACTIONS.index(DETAIL_TARGET_SIGNAL_FRACTION)
detail_snr = SNR_VALUES[snr_index]
detail_seed = BASE_SEED + 10_000 * snr_index + DETAIL_REPLICATE
detail_optimizer_seed = detail_seed + LAMBDA1_VALUES.index(DETAIL_LAMBDA1) + 1

detail = fit_beta2_roc_detail(
    data_dir=PARAMS['DATA_DIR'], p=PARAMS['P'], n=PARAMS['N_SUBJECTS'],
    beta1_true=beta1_true, beta2_true=beta2_true, snr=detail_snr,
    seed=detail_seed, optimizer_seed=detail_optimizer_seed,
    lambda1=DETAIL_LAMBDA1, lambda2=FIXED_LAMBDA2,
    max_iter=MAX_ITER, step_size=PARAMS['STEP_SIZE'], tol=PARAMS['TOL'],
    num_candidates=PARAMS['NUM_CANDIDATES'],
    initialization=PARAMS['INITIALIZATION'],
)
detail_path = run_dir / 'figures' / (
    f'beta_overlay_target_{DETAIL_TARGET_SIGNAL_FRACTION:g}_'
    f'rep_{DETAIL_REPLICATE + 1}_lambda1_{DETAIL_LAMBDA1:g}.png'
)
fig, axes = plot_beta2_roc_detail(detail, save_path=detail_path)
print(f'Saved detail figure to {detail_path}')
plt.show()
